# Data Loading and Preparation

In [ ]:
# Basic imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look nicer
sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
# Load raw UNSW-NB15 datasets and feature names
from pathlib import Path

feat_path = Path("../data/raw/UNSW-NB15_features.csv")
data_path1 = Path("../data/raw/UNSW-NB15_1.csv")
data_path2 = Path("../data/raw/UNSW-NB15_2.csv")

# Read the features file
feat_df = pd.read_csv(feat_path, encoding="ISO-8859-1")
feat_df.head()

In [ ]:
# Combine UNSW-NB15 CSVs into a single DataFrame and inspect structure
col_names = feat_df["Name"].tolist()
df1 = pd.read_csv(data_path1, names=col_names, low_memory=False)
df2 = pd.read_csv(data_path2, names=col_names, low_memory=False)
df_raw = pd.concat([df1, df2], ignore_index=True) # Ensures unique indices across files

# Show first few rows
display(df_raw.head())

# Check shape
print("Dataset shape:", df_raw.shape)

# Info about data types and missing values
df_raw.info()

# Quick statistics for numeric columns
display(df_raw.describe())

In [ ]:
# Prepare dataset for EDA and visualisation
df_eda = df_raw.copy() # Use the full dataset

# Show first few rows
display(df_eda.head())

# Check shape
print("Sample shape:", df_eda.shape)

In [ ]:
# Visualise traffic class balance
sns.countplot(x="Label", data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Count")
plt.title("Normal vs Attack Traffic")
plt.show()

# Byte-Based Features

Visualising byte distributions to understand volume patterns.

### Source and Destination Byte Distributions

In [ ]:
# Compare distribution of source bytes across traffic types (log-scaled)
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=np.log1p(df_eda["sbytes"]), data=df_eda) # Apply scaling inline for visualisation only
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Source Bytes + 1)")
plt.title("Distribution of Log(Source Bytes) by Label")
plt.show()

In [ ]:
# Compare distribution of destination bytes across traffic types (log-scaled)
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=np.log1p(df_eda["dbytes"]), data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Destination Bytes + 1)")
plt.title("Distribution of Log(Destination Bytes) by Label")
plt.show()

In [ ]:
# Scatter scaled relationship between source and destination bytes
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=np.log1p(df_eda["sbytes"]),
    y=np.log1p(df_eda["dbytes"]),
    data=df_eda,
    hue="Label",
    alpha=0.6,
    palette={0: "blue", 1: "red"}
)
plt.xlabel("Log(Source Bytes + 1)")
plt.ylabel("Log(Destination Bytes + 1)")
plt.title("Log(Source Bytes) vs Log(Destination Bytes) by Label")
plt.show()

### Derived Byte Distributions

In [ ]:
# Highlight flows with directionally imbalanced volumes
log_byte_ratio = np.log1p(df_eda["sbytes"]/(df_eda["dbytes"] + 1))
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=log_byte_ratio, data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Byte Ratio + 1)")
plt.title("Distribution of Log(Byte Ratio) by Label")
plt.show()

In [ ]:
# Plot density distribution of log-scaled byte ratio by label
plt.figure(figsize=(8, 6))
sns.histplot(
    data=df_eda,
    x=log_byte_ratio,
    hue="Label",
    bins=100,
    element="step",
    stat="density",
    common_norm=False,
    alpha=0.6,
    palette={0: "blue", 1: "red"}
)
plt.xlabel("Log(Byte Ratio + 1)")
plt.ylabel("Density")
plt.title("Distribution of Byte Asymmetry (Source/Destination Bytes)")
plt.show()

In [ ]:
# Highlight unusual byte combinations
abs_log_byte_diff = abs(np.log1p(df_eda["sbytes"]) - np.log1p(df_eda["dbytes"]))
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=abs_log_byte_diff, data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Abs(Log(Byte Difference + 1))")
plt.title("Distribution of Abs(Log(Byte Difference)) by Label")
plt.show()

In [ ]:
# Downweight bidirectional network flows
log_min_flow_bytes = np.log1p(np.minimum(df_eda["sbytes"], df_eda["dbytes"]))
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=log_min_flow_bytes, data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Min(Source Bytes, Destination Bytes) + 1)")
plt.title("Distribution of Log(Minimum Flow Bytes) by Label")
plt.show()

In [ ]:
# Explore combined flow volume
log_byte_prod = np.log1p(df_eda["sbytes"]*df_eda["dbytes"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=log_byte_prod, data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Source Bytes*Destination Bytes + 1)")
plt.title("Distribution of Log(Byte Product) by Label")
plt.show()

# Time-Based Features

Visualising duration distributions to understand temporal patterns.

### Flow Duration Distributions

In [ ]:
# Compare distribution of duration across traffic types (log-scaled)
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=np.log1p(df_eda["dur"]), data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Duration + 1)")
plt.title("Distribution of Log(Duration) by Label")
plt.show()

In [ ]:
# Scatter scaled relationship between byte ratio and duration
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_eda,
    x=log_byte_ratio,
    y=np.log1p(df_eda["dur"]),
    hue="Label",
    alpha=0.6,
    palette={0: "blue", 1: "red"}
)
plt.xlabel("Log(Byte Ratio + 1)")
plt.ylabel("Log(Duration + 1)")
plt.title("Log(Byte Ratio) vs Log(Duration) by Label")
plt.show()

### Endpoint Load Distributions

In [ ]:
# Compare distribution of source load across traffic types (log-scaled)
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=np.log1p(df_eda["Sload"]), data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Source Load + 1)")
plt.title("Distribution of Log(Source Load) by Label")
plt.show()

In [ ]:
# Compare distribution of destination load across traffic types (log-scaled)
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=np.log1p(df_eda["Dload"]), data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Destination Load + 1)")
plt.title("Distribution of Log(Destination Load) by Label")
plt.show()

In [ ]:
# Highlight flows with directionally imbalanced loads
load_skew = np.log1p(df_eda["Dload"])/(np.log1p(df_eda["Sload"]) + 1e-6)
load_skew = load_skew.clip(upper=load_skew.quantile(0.999))
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=load_skew, data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Load Skew Log(Log(Destination Load)/Log(Source Load))")
plt.title("Distribution of Load Skew by Label")
plt.show()

### Endpoint Jitter Distributions

In [ ]:
# Compare distribution of source jitter across traffic types
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="Sjit", data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Source Jitter")
plt.title("Distribution of Source Jitter by Label")
plt.show()

In [ ]:
# Compare distribution of destination jitter across traffic types
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="Djit", data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Destination Jitter")
plt.title("Distribution of Destination Jitter by Label")
plt.show()

### Recurrence Count Distributions

In [ ]:
# Experiment with temporal repetition
temporal_recurrence = np.log1p(df_eda["ct_src_dport_ltm"]+df_eda["ct_dst_sport_ltm"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=temporal_recurrence, data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Temporal Recurrence")
plt.title("Distribution of Temporal Recurrence by Label")
plt.show()

# Packet-Level Features

Visualising size distributions to understand packet patterns.

### Source and Destination Mean Packet Size Distributions

In [ ]:
# Compare distribution of source mean packet size across traffic types (log-scaled)
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=np.log1p(df_eda["smeansz"]), data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Source Mean Packet Size + 1)")
plt.title("Distribution of Log(Source Mean Packet Size) by Label")
plt.show()

In [ ]:
# Compare distribution of destination mean packet size across traffic types (log-scaled)
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=np.log1p(df_eda["dmeansz"]), data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Destination Mean Packet Size + 1)")
plt.title("Distribution of Log(Destination Mean Packet Size) by Label")
plt.show()

### Derived Mean Packet Size Distributions

In [ ]:
# Highlight flows with directionally imbalanced mean packet sizes
mean_pkt_sz_ratio = np.log1p(df_eda["dmeansz"])/(np.log1p(df_eda["smeansz"]) + 1e-6)
mean_pkt_sz_ratio = mean_pkt_sz_ratio.clip(upper=mean_pkt_sz_ratio.quantile(0.999))
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=mean_pkt_sz_ratio, data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Mean Packet Size Ratio")
plt.title("Distribution of Mean Packet Size Ratio by Label")
plt.show()

# TTL Features

Visualising time-to-live distributions to understand hop patterns.

### Source and Destination TTL Distributions

In [ ]:
# Compare distribution of source ttl across traffic types
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="sttl", data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Source TTL")
plt.title("Distribution of Source TTL by Label")
plt.show()

In [ ]:
# Compare distribution of destination ttl across traffic types
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="dttl", data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Destination TTL")
plt.title("Distribution of Destination TTL by Label")
plt.show()

# Visualise bimodality
plt.figure(figsize=(8, 6))
sns.histplot(df_eda[df_eda["Label"]==1]["dttl"], bins=30, kde=False)
plt.title("Histogram of Destination TTL for Attacks")
plt.xlabel("Destination TTL")
plt.ylabel("Count")
plt.show()

In [ ]:
# Scatter relationship between source ttl and destination ttl
plt.figure(figsize=(8, 6))
sns.scatterplot(x="sttl", y="dttl", hue="Label", data=df_eda, alpha=0.6)
plt.xlabel("Source TTL")
plt.ylabel("Destination TTL")
plt.title("Source TTL vs Destination TTL by Label")
plt.show()

# Comparative histograms
plt.figure(figsize=(8, 6))
sns.kdeplot(df_eda[df_eda['Label']==0]['dttl'], label='Normal', bw_adjust=0.5)
sns.kdeplot(df_eda[df_eda['Label']==1]['dttl'], label='Attack', bw_adjust=0.5)
plt.title("TTL Distributions by Label")
plt.legend()
plt.show()

### Derived TTL Distributions

In [ ]:
# Highlight unusual ttl combinations
ttl_diff = abs(df_eda["sttl"] - df_eda["dttl"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y=ttl_diff, data=df_eda)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("TTL Difference")
plt.title("Distribution of TTL Difference by Label")
plt.show()

### Checking Feature Correlation with Label

In [ ]:
# Data leakage
pd.crosstab(
    df_eda["ct_state_ttl"] > 0,
    df_eda["Label"],
    normalize="columns"
)

# Categorical Features

### Initial Exploration of Distributions

In [ ]:
# Identify and inspect categorical features
categorical_features = df_eda.select_dtypes(include=['object', 'category']).columns.tolist()
low_card_numeric = [col for col in df_eda.select_dtypes(include=[int, float]).columns if df_eda[col].nunique() <= 20]
categorical_features += low_card_numeric
print(categorical_features)

for col in ["sport", "dsport", "state", "service", "proto"]:
    print(f"\nValue counts for {col}:")
    print(df_eda[col].value_counts().head(10)) # Top 10 categories

In [ ]:
# Examine distribution of key categorical features for attack flows
attack_subset = df_eda[df_eda["Label"] == 1]
for col in ["state", "service", "proto"]:
    counts = attack_subset[col].value_counts()
    frac = counts / counts.sum()
    attack_df = pd.DataFrame({
        "attack_count": counts,
        "attack_frac": frac
    }).sort_values(by="attack_count", ascending=False)
    print(f"\nTop {col} values for attacks:")
    display(attack_df)

### Thorough Examination of Candidates

In [ ]:
# Compare distribution of "is_sm_ips_ports" across traffic types
attack_subset = df_eda[df_eda["Label"] == 1]
normal_subset = df_eda[df_eda["Label"] == 0]

attack_counts = attack_subset["is_sm_ips_ports"].value_counts().sort_index()
normal_counts = normal_subset["is_sm_ips_ports"].value_counts().sort_index()
print(attack_counts, normal_counts)

df_counts = pd.DataFrame({
    "Normal": normal_counts,
    "Attack": attack_counts
}).fillna(0)

# Plot as bar chart
df_counts.plot(kind="bar", figsize=(8, 6), color={"Normal": "blue", "Attack": "red"})
plt.title("Distribution of is_sm_ips_ports by Label")
plt.xlabel("is_sm_ips_ports")
plt.ylabel("Number of flows")
plt.xticks([0, 1], ["0 (unique)", "1 (repeated)"], rotation=0)
plt.show()

In [ ]:
# Compare distribution of transaction depth across traffic types
plt.figure(figsize=(8, 6))
sns.countplot(
    data=df_eda,
    x="trans_depth",
    hue="Label",
    palette={0: "blue", 1: "red"}
)
plt.title("Distribution of Transaction Depth by Label")
plt.xlabel("Transaction Depth")
plt.ylabel("Count")
plt.show()

display(
    df_eda[df_eda["Label"] == 0]["trans_depth"]
    .value_counts()
    .sort_index()
)

display(
    df_eda[df_eda["Label"] == 1]["trans_depth"]
    .value_counts()
    .sort_index()
)

In [ ]:
# Compare distribution of HTTP flow method count across traffic types
plt.figure(figsize=(8, 6))
sns.countplot(
    data=df_eda,
    x="ct_flw_http_mthd",
    hue="Label",
    palette={0: "blue", 1: "red"}
)
plt.title("Distribution of HTTP Flow Method Count by Label")
plt.xlabel("HTTP Flow Method Count")
plt.ylabel("Count")
plt.show()

display(
    df_eda[df_eda["Label"] == 0]["ct_flw_http_mthd"]
    .value_counts()
    .sort_index()
)

display(
    df_eda[df_eda["Label"] == 1]["ct_flw_http_mthd"]
    .value_counts()
    .sort_index()
)